# Filterbank RFI cleaning

This notebook writes a new SIGPROC `.fil` file and never modifies the source. 
Each complete 256-sample block is classified channel by channel with either the legacy-compatible 1D CNN or the saved scikit-learn MLP. 
The final incomplete block is copied unchanged.

`MODEL_KIND` and `REPLACEMENT` are included in the output filename. Before a full run, configure the paths and inspect the preview.
Writing is disabled until `RUN_CLEANING = True

> [!warning]
> For noise replacement, define trusted RFI-free half-open intervals `(start_sample, stop_sample)` in `CLEAN_REFERENCE_RANGES`.
> One Gaussian distribution is estimated from these original samples before cleaning starts.",
markdown",

The CNN uses the audited 256-sample architecture and maximum normalization separately for each channel. The selected MLP is intentionally restricted to its final three features: `mean_o`, `std_o`, and `skew_o`. The feature order is checked against the saved bundle before use.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from your import Your
from your.formats.pysigproc import SigprocFile


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "rfimt").is_dir():
            return candidate
    raise RuntimeError("Could not locate the rfimt repository.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from rfimt.models import CNN1DRFI256Logits


In [2]:
# Set these values before running the preparation cell below.
# INPUT_FILTERBANK = Path("/hercules/results/akazantsev/filterbank_files/B0531+21/B0531+21_59000_48386.fil")
INPUT_FILTERBANK = Path("/hercules/results/akazantsev/filterbank_files/B0531+21/B0531+21_60482_57794.fil")
OUTPUT_DIRECTORY = INPUT_FILTERBANK.parent / "rfi_cleaned"

MODEL_KIND = "mlp"             # "cnn" or "mlp"
REPLACEMENT = "zero"            # "zero" or "noise"
SEGMENT_NSAMP = 256
RANDOM_SEED = 42


In [3]:
# Required only for REPLACEMENT = "noise". Intervals are [start, stop).
CLEAN_REFERENCE_RANGES: list[tuple[int, int]] = [(0, 8192)]
REFERENCE_READ_NSAMP = 8192

In [4]:
# Set only the path required by MODEL_KIND.
CNN_CHECKPOINT = Path(
    "/u/akazantsev/rfimt/performance_comparison/outputs/performance_comparison/"
    "b0531_legacy_performance_v1/cnn_cuda_legacy_max/checkpoint.pt"
)

CNN_THRESHOLD = 0.50
CNN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MLP_BUNDLE = Path(
    "/u/akazantsev/rfimt/performance_comparison/outputs/performance_comparison/"
    "b0531_legacy_performance_v1/mlp_orig_top3/mlp_orig_top3.joblib"
)


if MODEL_KIND not in {"cnn", "mlp"}:
    raise ValueError("MODEL_KIND must be 'cnn' or 'mlp'.")
if REPLACEMENT not in {"zero", "noise"}:
    raise ValueError("REPLACEMENT must be 'zero' or 'noise'.")
if SEGMENT_NSAMP != 256:
    raise ValueError("The trained models require 256-sample channel profiles.")

model_tag = {"cnn": "1dcnn", "mlp": "mlp"}[MODEL_KIND]
output_stem = f"{INPUT_FILTERBANK.stem}_cleaned_{model_tag}_{REPLACEMENT}"
OUTPUT_FILTERBANK = OUTPUT_DIRECTORY / f"{output_stem}.fil"
SUMMARY_PATH = OUTPUT_DIRECTORY / f"{output_stem}_segment_summary.csv"
MASKS_PATH = OUTPUT_DIRECTORY / f"{output_stem}_masks.npz"
MANIFEST_PATH = OUTPUT_DIRECTORY / f"{output_stem}_manifest.json"

RUN_HEADER_PREFLIGHT = True
HEADER_PREFLIGHT_PATH = OUTPUT_DIRECTORY / f".{output_stem}_header_preflight.fil"

print(f"Repository: {REPO_ROOT}")
print(f"Output: {OUTPUT_FILTERBANK}")

Repository: /u/akazantsev/ml_dl_for_rfi_mitigation
Output: /hercules/results/akazantsev/filterbank_files/B0531+21/rfi_cleaned/B0531+21_60482_57794_cleaned_mlp_zero.fil


In [5]:
def _load_cleaned_preview_segment(segment_index: int, rng: np.random.Generator):
    """Read one complete segment, classify its channels, and apply the configured replacement."""
    if not 0 <= segment_index < n_complete_segments:
        raise ValueError(
            f"segment_index={segment_index} must be in "
            f"[0, {n_complete_segments - 1}]."
        )

    start = segment_index * SEGMENT_NSAMP
    original_ct = read_intensity_block(
        source_fil,
        start,
        SEGMENT_NSAMP,
        n_channels,
    ).T

    row_mask, probabilities = predict_segment(original_ct)
    cleaned_ct = replace_flagged(
        original_ct,
        row_mask,
        rng,
        noise_mean,
        noise_std,
    )
    return original_ct, cleaned_ct, row_mask, probabilities


def _draw_preview_pair(axes, original_ct, cleaned_ct, row_label: str):
    """Draw one original/cleaned row with one shared intensity scale."""
    vmin = float(np.min(original_ct))
    vmax = float(np.max(original_ct))

    for axis, values, title in zip(
        axes,
        (original_ct, cleaned_ct),
        ("Original", f"Cleaned ({REPLACEMENT})"),
    ):
        axis.imshow(
            values,
            aspect="auto",
            origin="upper",
            cmap="gray",
            vmin=vmin,
            vmax=vmax,
        )
        axis.set(title=title, xlabel="Time sample")

    axes[0].set_ylabel(f"{row_label}\nChannel")


def _finish_preview_figure(figure, output_name: str, save_figure: bool):
    figure.tight_layout()

    if save_figure:
        output_dir = Path("./imgs")
        output_dir.mkdir(parents=True, exist_ok=True)
        output_file = output_dir / output_name
        figure.savefig(output_file, dpi=150, bbox_inches="tight")
        plt.close(figure)
        print(f"Saved: {output_file}")
    else:
        plt.show()


def process_preview_segment(segment_index: int, save_figure: bool = False):
    """Render the original and cleaned forms of one named segment."""
    original_ct, cleaned_ct, row_mask, _ = _load_cleaned_preview_segment(
        segment_index,
        np.random.default_rng(RANDOM_SEED),
    )

    figure, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
    _draw_preview_pair(axes, original_ct, cleaned_ct, f"Segment {segment_index}")

    figure.suptitle(
        f"Segment {segment_index}; {MODEL_KIND}; "
        f"threshold={model_details['threshold']:.3f}"
    )
    print(f"Segment {segment_index}: Flagged channels: {row_mask.sum()} / {n_channels}")

    _finish_preview_figure(
        figure,
        f"{Path(INPUT_FILTERBANK).stem}_{segment_index}_{MODEL_KIND}_{REPLACEMENT}.png",
        save_figure,
    )


def plot_pulse_and_rfi_preview(
    pulse_index: int,
    rfi_index: int,
    *,
    rfi_label: str = "NBRFI",
    save_figure: bool = False,
):
    """Show a pulse and an RFI segment together as original/cleaned pairs."""
    rng = np.random.default_rng(RANDOM_SEED)
    examples = (
        ("Pulse", pulse_index),
        (rfi_label, rfi_index),
    )

    figure, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)
    flagged_counts = []

    for row_axes, (label, segment_index) in zip(axes, examples):
        original_ct, cleaned_ct, row_mask, _ = _load_cleaned_preview_segment(
            segment_index,
            rng,
        )
        _draw_preview_pair(row_axes, original_ct, cleaned_ct, f"{label} {segment_index}")
        flagged_counts.append((label, segment_index, int(row_mask.sum())))

    figure.suptitle(
        f"Pulse {pulse_index} and {rfi_label} {rfi_index}; "
        f"{MODEL_KIND}; threshold={model_details['threshold']:.3f}"
    )
    print(
        " | ".join(
            f"{label} {segment_index}: {count} / {n_channels} flagged"
            for label, segment_index, count in flagged_counts
        )
    )

    _finish_preview_figure(
        figure,
        (
            f"{Path(INPUT_FILTERBANK).stem}_pulse_{pulse_index}_"
            f"{rfi_label.lower()}_{rfi_index}_{MODEL_KIND}_{REPLACEMENT}.png"
        ),
        save_figure,
    )


def plot_shift_invariance_preview(
    segment_index: int,
    *,
    n_shifts: int = 1,
    random_seed: int | None = RANDOM_SEED,
    save_figure: bool = False,
):
    """
    Reclassify and clean circularly shifted copies of one original segment.

    Each shift starts from the same original array, so the plot isolates the
    effect of translating the input along channel and time axes.
    """
    if not isinstance(n_shifts, int) or n_shifts < 1:
        raise ValueError("n_shifts must be a positive integer.")

    rng = np.random.default_rng(random_seed)
    original_ct, cleaned_ct, row_mask, _ = _load_cleaned_preview_segment(
        segment_index,
        rng,
    )

    figure, axes = plt.subplots(
        n_shifts + 1,
        2,
        figsize=(12, 5 * (n_shifts + 1)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    _draw_preview_pair(axes[0], original_ct, cleaned_ct, f"Original {segment_index}")

    shift_descriptions = []
    for shift_number in range(1, n_shifts + 1):
        channel_shift = int(rng.integers(1, n_channels)) if n_channels > 1 else 0
        time_shift = (
            int(rng.integers(1, SEGMENT_NSAMP)) if SEGMENT_NSAMP > 1 else 0
        )
        shifted_ct = np.roll(
            original_ct,
            shift=(channel_shift, time_shift),
            axis=(0, 1),
        )
        shifted_mask, shifted_probabilities = predict_segment(shifted_ct)
        shifted_cleaned_ct = replace_flagged(
            shifted_ct,
            shifted_mask,
            rng,
            noise_mean,
            noise_std,
        )

        _draw_preview_pair(
            axes[shift_number],
            shifted_ct,
            shifted_cleaned_ct,
            f"Shift {shift_number}: Δchannel={channel_shift}, Δtime={time_shift}",
        )
        shift_descriptions.append(
            (
                shift_number,
                channel_shift,
                time_shift,
                int(shifted_mask.sum()),
                float(shifted_probabilities.mean()),
            )
        )

    figure.suptitle(
        f"Shift-invariance preview: segment {segment_index}; {MODEL_KIND}; "
        f"threshold={model_details['threshold']:.3f}; seed={random_seed}"
    )
    print(f"Original {segment_index}: {int(row_mask.sum())} / {n_channels} flagged")
    for shift_number, channel_shift, time_shift, flagged, mean_probability in shift_descriptions:
        print(
            f"Shift {shift_number}: Δchannel={channel_shift}, Δtime={time_shift}; "
            f"{flagged} / {n_channels} flagged; "
            f"mean probability={mean_probability:.4f}"
        )

    _finish_preview_figure(
        figure,
        (
            f"{Path(INPUT_FILTERBANK).stem}_{segment_index}_"
            f"{MODEL_KIND}_{REPLACEMENT}_shift_invariance.png"
        ),
        save_figure,
    )


In [6]:
def make_top3_features(segment_ct: np.ndarray) -> pd.DataFrame:
    values = np.asarray(segment_ct, dtype=np.float64)
    mean_o = values.mean(axis=1)
    std_o = values.std(axis=1)
    centered = values - mean_o[:, None]
    safe_std = np.where(std_o == 0, 1.0, std_o)
    skew_o = (centered ** 3).mean(axis=1) / safe_std ** 3
    return pd.DataFrame({"mean_o": mean_o, "std_o": std_o, "skew_o": skew_o})


def load_predictor():
    if MODEL_KIND == "cnn":
        if CNN_CHECKPOINT is None or not CNN_CHECKPOINT.is_file():
            raise FileNotFoundError("Set CNN_CHECKPOINT to an existing checkpoint.")
        device = torch.device(CNN_DEVICE)
        if device.type == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CNN_DEVICE is 'cuda', but CUDA is unavailable.")
        checkpoint = torch.load(CNN_CHECKPOINT, map_location=device)
        model = CNN1DRFI256Logits(dropout=0.5).to(device)
        model.load_state_dict(checkpoint.get("model_state_dict", checkpoint))
        model.eval()

        @torch.no_grad()
        def predict(segment_ct: np.ndarray):
            values = np.asarray(segment_ct, dtype=np.float32)
            maximum = values.max(axis=1, keepdims=True)
            maximum = np.where(maximum < 1e-8, 1.0, maximum)
            batch = torch.from_numpy(values / maximum).unsqueeze(1).to(device)
            probabilities = torch.sigmoid(model(batch)).cpu().numpy().astype(np.float32)
            return probabilities >= CNN_THRESHOLD, probabilities

        return predict, {
            "model_kind": "cnn", "checkpoint": str(CNN_CHECKPOINT),
            "threshold": float(CNN_THRESHOLD), "device": str(device),
            "normalization": "maximum per channel",
        }

    if MLP_BUNDLE is None or not MLP_BUNDLE.is_file():
        raise FileNotFoundError("Set MLP_BUNDLE to an existing MLP bundle.")
    bundle = joblib.load(MLP_BUNDLE)
    expected_features = ["mean_o", "std_o", "skew_o"]
    if list(bundle["feature_cols"]) != expected_features:
        raise ValueError(
            f"This cleaner expects {expected_features}; bundle has {bundle['feature_cols']}."
        )
    pipeline = bundle["pipeline"]
    threshold = float(bundle["threshold"])

    def predict(segment_ct: np.ndarray):
        probabilities = pipeline.predict_proba(make_top3_features(segment_ct))[:, 1]
        probabilities = probabilities.astype(np.float32)
        return probabilities >= threshold, probabilities

    return predict, {
        "model_kind": "mlp", "bundle": str(MLP_BUNDLE),
        "threshold": threshold, "feature_cols": expected_features,
    }


## Prepare the filterbank and the replacement distribution

`Your.get_data` supplies time-by-channel data. It is transposed only for model inference. For noise replacement, the stated clean regions are read from the original file and pooled into a single mean and standard deviation.

In [7]:
def require_input_filterbank() -> None:
    if str(INPUT_FILTERBANK) == "/path/to/input.fil":
        raise ValueError("Set INPUT_FILTERBANK before running this cell.")
    if INPUT_FILTERBANK.suffix.lower() != ".fil" or not INPUT_FILTERBANK.is_file():
        raise FileNotFoundError(INPUT_FILTERBANK)


def read_intensity_block(
    filterbank: SigprocFile,
    start: int,
    nsamp: int,
    n_channels: int,
) -> np.ndarray:
    """Read one nifs=1 block as (time sample, channel)."""
    block = np.asarray(filterbank.get_data(start, nsamp))
    if block.ndim != 3 or block.shape[1] != 1 or block.shape[2] != n_channels:
        raise ValueError(
            "Expected SigprocFile.get_data to return "
            f"(n_samples, 1, {n_channels}), received {block.shape}."
        )
    return np.ascontiguousarray(block[:, 0, :])


def estimate_reference_noise(
    filterbank: SigprocFile,
    n_channels: int,
    n_total_samples: int,
) -> tuple[float, float]:
    if not CLEAN_REFERENCE_RANGES:
        raise ValueError("Noise replacement requires CLEAN_REFERENCE_RANGES.")
    total_count = total_sum = total_sum_sq = 0.0
    for start, stop in CLEAN_REFERENCE_RANGES:
        if not (0 <= start < stop <= n_total_samples):
            raise ValueError(
                f"Invalid clean range {(start, stop)} for {n_total_samples} samples."
            )
        for offset in range(start, stop, REFERENCE_READ_NSAMP):
            count = min(REFERENCE_READ_NSAMP, stop - offset)
            values = np.asarray(
                read_intensity_block(filterbank, offset, count, n_channels),
                dtype=np.float64,
            )
            total_count += values.size
            total_sum += values.sum()
            total_sum_sq += np.square(values).sum()
    mean = total_sum / total_count
    std = np.sqrt(max(total_sum_sq / total_count - mean ** 2, 0.0))
    return float(mean), max(float(std), 1e-6)


def cast_like_input(values: np.ndarray, dtype: np.dtype) -> np.ndarray:
    if np.issubdtype(dtype, np.integer):
        limits = np.iinfo(dtype)
        values = np.clip(np.rint(values), limits.min, limits.max)
    return values.astype(dtype, copy=False)


HEADER_COMPARISON_FIELDS = (
    "nchans", "nbits", "nifs", "tsamp", "fch1", "foff", "tstart",
)


def build_output_header(source_header: SigprocFile, output_path: Path) -> SigprocFile:
    """Copy a SIGPROC header using the same byte-to-text fix as the I/O test."""
    output_header = SigprocFile(copy_hdr=source_header)
    for field_name, field_type in output_header._type.items():
        value = getattr(output_header, field_name)
        if field_type == "string" and isinstance(value, bytes):
            setattr(output_header, field_name, value.decode("utf-8"))
    output_header.rawdatafile = output_path.name
    return output_header


def validate_header_round_trip(source_header: SigprocFile) -> None:
    """Verify the output-header procedure before full-file processing."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    if HEADER_PREFLIGHT_PATH.exists():
        HEADER_PREFLIGHT_PATH.unlink()

    output_header = build_output_header(source_header, HEADER_PREFLIGHT_PATH)
    try:
        output_header.write_header(str(HEADER_PREFLIGHT_PATH))
        checked_header = SigprocFile(str(HEADER_PREFLIGHT_PATH))
        mismatches = [
            field_name for field_name in HEADER_COMPARISON_FIELDS
            if getattr(source_header, field_name) != getattr(checked_header, field_name)
        ]
        if mismatches:
            raise AssertionError(f"Header fields changed during round trip: {mismatches}")
    finally:
        if HEADER_PREFLIGHT_PATH.exists():
            HEADER_PREFLIGHT_PATH.unlink()

    print("Header round-trip preflight passed.")


def replace_flagged(segment_ct, row_mask, rng, noise_mean, noise_std):
    cleaned = np.asarray(segment_ct).copy()
    if not row_mask.any():
        return cleaned
    if REPLACEMENT == "zero":
        cleaned[row_mask, :] = 0
        return cleaned
    noise = rng.normal(noise_mean, noise_std, size=(int(row_mask.sum()), cleaned.shape[1]))
    cleaned[row_mask, :] = cast_like_input(noise, cleaned.dtype)
    return cleaned


require_input_filterbank()
source_fil = SigprocFile(str(INPUT_FILTERBANK))

n_channels = int(source_fil.nchans)
n_ifs = int(source_fil.nifs)
n_total_samples = int(source_fil.native_nspectra())
if n_ifs != 1:
    raise NotImplementedError(
        f"This cleaner currently supports nifs=1, received nifs={n_ifs}."
    )
if int(source_fil.nbits) not in {8, 16, 32}:
    raise NotImplementedError("Supported input bit depths are 8, 16, and 32.")

n_complete_segments, tail_nsamp = divmod(n_total_samples, SEGMENT_NSAMP)
if RUN_HEADER_PREFLIGHT:
    validate_header_round_trip(source_fil)

probe_tc = read_intensity_block(
    source_fil, 0, min(SEGMENT_NSAMP, n_total_samples), n_channels
)
input_dtype = probe_tc.dtype
predict_segment, model_details = load_predictor()
probe_mask, probe_probabilities = predict_segment(probe_tc.T)
if probe_mask.shape != (n_channels,) or probe_probabilities.shape != (n_channels,):
    raise ValueError("The model must return one mask entry and probability per channel.")

noise_mean = noise_std = None
if REPLACEMENT == "noise":
    noise_mean, noise_std = estimate_reference_noise(
        source_fil, n_channels, n_total_samples
    )

print(f"Samples: {n_total_samples}; channels: {n_channels}; dtype: {input_dtype}")
print(f"Complete blocks: {n_complete_segments}; unchanged tail: {tail_nsamp} samples")
print(f"Probe mask: {probe_mask.sum()} / {n_channels} channels")
if REPLACEMENT == "noise":
    print(f"Reference noise: mean={noise_mean:.6g}, std={noise_std:.6g}")


Header round-trip preflight passed.
Samples: 44000000; channels: 256; dtype: uint8
Complete blocks: 171875; unchanged tail: 0 samples
Probe mask: 117 / 256 channels


## Preview

Before allowing a full write, inspect paired pulse/RFI figures: each has original and cleaned panels for both examples. The shift-invariance figure starts from one original segment, circularly shifts it independently along the channel and time axes, then classifies and cleans every shifted copy again.


In [8]:
PREVIEW_SEGMENT_INDEX = 4012
PREVIEW_SHIFT_COUNT = 2

filterbank_stem = Path(INPUT_FILTERBANK).stem

if filterbank_stem == "B0531+21_59000_48386":
    list_preview_segments_pulses = [15602, 40301, 17623]
    list_preview_segments_bbrfi = [9244, 15805, 25873]
    list_preview_segments_rfi = list_preview_segments_bbrfi
    preview_rfi_label = "BBRFI"

elif filterbank_stem == "B0531+21_60482_57794":
    list_preview_segments_pulses = [1017, 1327, 1328, 2686]
    list_preview_segments_bbrfi = [1072, 4012, 13998, 15200]
    list_preview_segments_none = [1404, 6400, 1366, 2474]
    list_preview_segments_nbrfi = [2552, 1366, 2474, 4506, 4757, 9456]
    list_preview_segments_rfi = list_preview_segments_nbrfi
    preview_rfi_label = "NBRFI"

else:
    raise ValueError(f"Unknown filterbank: {filterbank_stem}")


In [9]:
for pulse_index, rfi_index in zip(
    list_preview_segments_pulses,
    list_preview_segments_rfi,
):
    plot_pulse_and_rfi_preview(
        pulse_index,
        rfi_index,
        rfi_label=preview_rfi_label,
        save_figure=True,
    )

plot_shift_invariance_preview(
    PREVIEW_SEGMENT_INDEX,
    n_shifts=PREVIEW_SHIFT_COUNT,
    random_seed=RANDOM_SEED,
    save_figure=True,
)


In [10]:
# The paired pulse/RFI and shift-invariance previews are generated above.


Segment 1017: Flagged channels: 255 / 256
Saved: imgs/B0531+21_60482_57794_1017_mlp_zero.png
Segment 1327: Flagged channels: 255 / 256
Saved: imgs/B0531+21_60482_57794_1327_mlp_zero.png
Segment 1328: Flagged channels: 254 / 256
Saved: imgs/B0531+21_60482_57794_1328_mlp_zero.png
Segment 2686: Flagged channels: 5 / 256
Saved: imgs/B0531+21_60482_57794_2686_mlp_zero.png
Segment 1072: Flagged channels: 232 / 256
Saved: imgs/B0531+21_60482_57794_1072_mlp_zero.png
Segment 4012: Flagged channels: 14 / 256
Saved: imgs/B0531+21_60482_57794_4012_mlp_zero.png
Segment 13998: Flagged channels: 6 / 256
Saved: imgs/B0531+21_60482_57794_13998_mlp_zero.png
Segment 15200: Flagged channels: 15 / 256
Saved: imgs/B0531+21_60482_57794_15200_mlp_zero.png
Segment 1404: Flagged channels: 248 / 256
Saved: imgs/B0531+21_60482_57794_1404_mlp_zero.png
Segment 6400: Flagged channels: 11 / 256
Saved: imgs/B0531+21_60482_57794_6400_mlp_zero.png
Segment 1366: Flagged channels: 255 / 256
Saved: imgs/B0531+21_60482_5779

## Write the cleaned filterbank

The source header is copied to the new output. Existing outputs are refused, and the source file is never opened for writing. The summary, masks, and run manifest are written beside the cleaned filterbank.

In [11]:
RUN_CLEANING = True

In [12]:
if not RUN_CLEANING:
    print("Full cleaning is disabled. Inspect the preview, then set RUN_CLEANING = True.")
else:
    if OUTPUT_FILTERBANK.resolve() == INPUT_FILTERBANK.resolve():
        raise ValueError("Output must not be the source filterbank.")
    outputs = (OUTPUT_FILTERBANK, SUMMARY_PATH, MASKS_PATH, MANIFEST_PATH)
    existing = [path for path in outputs if path.exists()]
    if existing:
        raise FileExistsError("Refusing to overwrite:\n" + "\n".join(map(str, existing)))

    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_header = build_output_header(source_fil, OUTPUT_FILTERBANK)
    output_header.write_header(str(OUTPUT_FILTERBANK))

    rng = np.random.default_rng(RANDOM_SEED)
    masks = np.empty((n_complete_segments, n_channels), dtype=bool)
    summary_rows = []

    for segment_index in tqdm(range(n_complete_segments), desc="Cleaning"):
        start = segment_index * SEGMENT_NSAMP
        original_tc = read_intensity_block(
            source_fil, start, SEGMENT_NSAMP, n_channels
        )
        original_ct = original_tc.T
        row_mask, probabilities = predict_segment(original_ct)
        row_mask = np.asarray(row_mask, dtype=bool)
        probabilities = np.asarray(probabilities, dtype=np.float32)
        cleaned_ct = replace_flagged(original_ct, row_mask, rng, noise_mean, noise_std)
        output_header.append_spectra(
            np.ascontiguousarray(cleaned_ct.T), str(OUTPUT_FILTERBANK)
        )
        masks[segment_index] = row_mask
        summary_rows.append({
            "segment_index": segment_index, "start_sample": start,
            "n_masked": int(row_mask.sum()), "fraction_masked": float(row_mask.mean()),
            "probability_min": float(probabilities.min()),
            "probability_mean": float(probabilities.mean()),
            "probability_max": float(probabilities.max()),
        })

    if tail_nsamp:
        tail_tc = read_intensity_block(
            source_fil, n_complete_segments * SEGMENT_NSAMP, tail_nsamp, n_channels
        )
        output_header.append_spectra(np.ascontiguousarray(tail_tc), str(OUTPUT_FILTERBANK))

    pd.DataFrame(summary_rows).to_csv(SUMMARY_PATH, index=False)
    np.savez_compressed(MASKS_PATH, masks=masks, segment_nsamp=SEGMENT_NSAMP)
    MANIFEST_PATH.write_text(json.dumps({
        "input_filterbank": str(INPUT_FILTERBANK),
        "output_filterbank": str(OUTPUT_FILTERBANK), "model": model_details,
        "replacement": REPLACEMENT, "clean_reference_ranges": CLEAN_REFERENCE_RANGES,
        "noise_mean": noise_mean, "noise_std": noise_std, "random_seed": RANDOM_SEED,
        "segment_nsamp": SEGMENT_NSAMP, "tail_nsamp_copied_unchanged": tail_nsamp,
    }, indent=2), encoding="utf-8")
    print(f"Created {OUTPUT_FILTERBANK}")
    print(f"Masked fraction: {masks.mean():.4%}")


Cleaning: 100%|██████████| 171875/171875 [1:34:16<00:00, 30.39it/s]   


Created /hercules/results/akazantsev/filterbank_files/B0531+21/rfi_cleaned/B0531+21_60482_57794_cleaned_mlp_zero.fil
Masked fraction: 19.9946%


## Validate the written output

Run this after cleaning. It reopens the output, checks defining header values and total sample count, and confirms that unmasked channels in the preview block are bitwise unchanged.

In [22]:
if not OUTPUT_FILTERBANK.is_file():
    print("No output filterbank exists yet.")
elif not MASKS_PATH.is_file():
    raise FileNotFoundError(
        f"The output exists, but its saved mask file is missing: {MASKS_PATH}"
    )
else:
    output_reader = Your(str(OUTPUT_FILTERBANK))
    print("Your successfully opened the cleaned filterbank.")

    written_fil = SigprocFile(str(OUTPUT_FILTERBANK))
    for field_name in HEADER_COMPARISON_FIELDS:
        if getattr(source_fil, field_name) != getattr(written_fil, field_name):
            raise AssertionError(f"Header mismatch for {field_name}.")
    if int(written_fil.native_nspectra()) != n_total_samples:
        raise AssertionError("Output sample count does not match the source.")

    with np.load(MASKS_PATH, allow_pickle=False) as saved_masks:
        saved_segment_nsamp = int(saved_masks["segment_nsamp"])
        masks_used_for_cleaning = np.asarray(saved_masks["masks"], dtype=bool)

    if saved_segment_nsamp != SEGMENT_NSAMP:
        raise AssertionError("Saved masks use a different segment size.")
    if masks_used_for_cleaning.shape != (n_complete_segments, n_channels):
        raise AssertionError(
            "Saved mask array does not match the cleaned filterbank dimensions."
        )

    mask_used_for_preview = masks_used_for_cleaning[PREVIEW_SEGMENT_INDEX]
    if not np.array_equal(mask_used_for_preview, preview_mask):
        print(
            "Note: the current in-memory preview mask differs from the mask used "
            "for this output. Validation uses the saved output mask."
        )

    written_tc = read_intensity_block(
        written_fil, preview_start, SEGMENT_NSAMP, n_channels
    )
    unmasked = ~mask_used_for_preview
    if not np.array_equal(written_tc[:, unmasked], preview_tc[:, unmasked]):
        changed_channels = np.flatnonzero(
            np.any(written_tc[:, unmasked] != preview_tc[:, unmasked], axis=0)
        )
        raise AssertionError(
            "Unmasked preview channels changed. Positions within the unmasked "
            f"selection: {changed_channels[:10].tolist()}"
        )
    if REPLACEMENT == "zero" and not np.all(written_tc[:, mask_used_for_preview] == 0):
        raise AssertionError("At least one masked preview channel was not zeroed.")

    print(
        "Validation passed: output opens with Your, header and sample count agree, "
        "and the preview replacement is correct."
    )


Your successfully opened the cleaned filterbank.


NameError: name 'preview_start' is not defined